|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 8:</h2>|<h1>The Capstone<h1>|
|<h2>Section:</h2>|<h1>Graphs on the real step<h1>|
|<h2>Lecture:</h2>|<h1><b>A thousand launches, not a thousand bytes<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# A thousand launches, not a thousand bytes

At batch 1, a decode step must read every weight one time. That sets a floor:
weight bytes divided by read bandwidth. Measure how far the eager step is
from that floor, and where the rest of the time goes.

In [2]:
from tvllm import load_model
model = load_model()
config = model.config
bandwidth = cudalib.read_bandwidth()
floor_ms = model.weight_bytes() / bandwidth * 1e3
print(f'read bandwidth {bandwidth/1e9:.0f} GB/s -> weight-read floor {floor_ms:.2f} ms')

/home/venugopalan/vllm-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 9842.96it/s]

read bandwidth 326 GB/s -> weight-read floor 3.65 ms


In [3]:
class StaticBackend:
    """A dense cache that a CUDA graph can capture. Each layer has a fixed
    (batch, kv_heads, max_len, head_dim) buffer, and a (batch,) tensor holds
    the positions. Each row of the batch is one sequence. A step does no
    allocation and no host sync."""
    def __init__(self, config, batch, max_len):
        shape = (batch, config.num_kv_heads, max_len, config.head_dim)
        self.key_cache = [torch.zeros(shape, dtype=config.dtype, device='cuda')
                          for _ in range(config.num_layers)]
        self.value_cache = [torch.zeros_like(cache) for cache in self.key_cache]
        self.positions = torch.zeros(batch, dtype=torch.long, device='cuda')
        self.batch_rows = torch.arange(batch, device='cuda')
        self.key_positions = torch.arange(max_len, device='cuda')

    def __call__(self, layer, query, key, value):
        batch, num_heads, head_dim = query.shape
        keys, values = self.key_cache[layer], self.value_cache[layer]
        keys[self.batch_rows, :, self.positions] = key
        values[self.batch_rows, :, self.positions] = value
        num_kv_heads = keys.shape[1]
        # Two grouped matmuls. SDPA with a mask and GQA uses a path that
        # copies K and V, and that copy costs more than the whole step.
        grouped_query = query.view(batch, num_kv_heads, num_heads // num_kv_heads, head_dim)
        scores = torch.matmul(grouped_query, keys.transpose(-1, -2)) * head_dim ** -0.5
        visible = self.key_positions[None, :] <= self.positions[:, None]
        scores = scores.masked_fill(~visible[:, None, None, :], float('-inf'))
        output = torch.matmul(scores.softmax(-1).to(values.dtype), values)
        return output.reshape(batch, num_heads * head_dim)

def decode_step(backend, batch):
    """-> a function that runs one decode step of the batch on backend."""
    token_ids = torch.zeros(batch, dtype=torch.long, device='cuda')
    logits_rows = torch.arange(batch, device='cuda')
    return lambda: model.forward(token_ids, backend.positions, backend, logits_rows)

### Count the kernels in one step

In [4]:
from torch.profiler import profile, ProfilerActivity
MAX_LEN = 256
backend = StaticBackend(config, batch=1, max_len=MAX_LEN)
backend.positions.fill_(40)
step = decode_step(backend, batch=1)
step()
torch.cuda.synchronize()
with profile(activities=[ProfilerActivity.CUDA]) as profiler:
    step()
    torch.cuda.synchronize()
num_kernels = sum(1 for event in profiler.events() if event.device_type.name == 'CUDA')
print(f'{num_kernels} GPU kernels in one decode step, '
      f'{num_kernels / config.num_layers:.0f} in each layer')

1076 GPU kernels in one decode step, 38 in each layer


USDT:2026-09-20 02:39:21 461420:461420 SyncActivityProfilerHandler.cpp:52] profiler_start
USDT:2026-09-20 02:39:21 461420:461420 SyncActivityProfilerHandler.cpp:59] profiler_stop


### Eager against graphed, against the floor

A CUDA graph replays all of those kernels with one call. The CPU no longer
issues them one at a time. Time both at three batch sizes.

In [5]:
def graphed(step):
    """Capture step() one time. -> a function that replays it."""
    side_stream = torch.cuda.Stream()
    side_stream.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(side_stream):
        for _ in range(3):
            step()
    torch.cuda.current_stream().wait_stream(side_stream)
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        output = step()
    def replay():
        graph.replay()
        return output
    return replay

rows = []
for batch in (1, 4, 16):
    backend = StaticBackend(config, batch, MAX_LEN)
    backend.positions.fill_(40)
    step = decode_step(backend, batch)
    eager_ms = cudalib.bench_ms(step, iters=30, best_of=3)
    replay = graphed(step)
    graph_ms = cudalib.bench_ms(replay, iters=30, best_of=3)
    rows.append((batch, eager_ms, graph_ms))
    del backend, replay
    torch.cuda.empty_cache()
print(f'{"batch":>5} {"eager / graph":>14} {"floor / eager":>14} {"floor / graph":>14}')
for batch, eager_ms, graph_ms in rows:
    print(f'{batch:>5} {eager_ms / graph_ms:>13.2f}x {100 * floor_ms / eager_ms:>13.0f}% '
          f'{100 * floor_ms / graph_ms:>13.0f}%')

batch  eager / graph  floor / eager  floor / graph
    1          2.61x            31%            81%
    4          2.26x            29%            65%
   16          1.72x            32%            55%


### What the three columns say

- **eager / graph** is the time that the CPU spent to issue kernels. A fast
  desktop CPU hides much of it. A slow cloud CPU hides little, and there the
  ratio is much larger. That is why this column changes from machine to
  machine, and the next two columns change much less.
- **floor / graph** is how close one step gets to the weight read. The rest is
  the small kernels between the big matmuls. They each move few bytes, and
  each one still costs a few microseconds.
- As the batch grows, eager / graph falls. The launch cost is the same for
  one row or sixteen, so it matters less next to more work.
- floor / graph also falls, for a reason that belongs to this demo. The floor
  here counts only the weights. This dense cache reads all 256 positions of
  every row, even where the mask hides them. The paged kernel of stage 08c
  reads only the real context, and the stage 25 floor counts those reads.

# The padding trap

A graph has a fixed batch, so a smaller step pads up to the bucket. Here is
what a padding row does in a PAGED cache. The demo uses plain tensors: a pool
of 4 blocks of 4 slots, and three real sequences in a bucket of 4.

In [6]:
pool = torch.zeros(16)
pool[0:3] = torch.tensor([7., 7., 7.])      # sequence A owns block 0
real_slots = torch.tensor([5, 9, 3])        # this step writes these slots
values = torch.tensor([1., 2., 3.])

def write(pool, slots, new_values):
    keep = slots >= 0                       # the write skips a slot of -1
    pool[slots[keep]] = new_values[keep]

def write_with_padding(padding_slot):
    """Write the real slots and one padding row. -> the pool after it."""
    result = pool.clone()
    write(result, torch.cat([real_slots, torch.tensor([padding_slot])]),
          torch.cat([values, torch.tensor([99.])]))
    return result

print('block 0, padding slot 0 :', write_with_padding(0)[:4].tolist())
print('block 0, padding slot -1:', write_with_padding(-1)[:4].tolist())

block 0, padding slot 0 : [99.0, 7.0, 7.0, 3.0]
block 0, padding slot -1: [7.0, 7.0, 7.0, 3.0]


### A zero is not a harmless value

With a padding slot of 0, the padding row overwrote position 0 of sequence A.
Nothing fails. Sequence A just attends to a wrong key from then on, and its
output changes a few tokens later.

In a dense cache each row owns its own memory, so the zeros of stage 12 were
safe. In a paged cache, every row writes into one shared pool, and slot 0 is
real. Stage 23 makes you pad with slot -1, which the stage 08 kernel skips.

    ./vc guide 23